# Figshare FoG Detection — Improved Pipeline
## Part 2: LOSO Evaluation Pipeline

Compares 6 classifiers using Leave-One-Subject-Out cross-validation:
- RandomForest, LogisticRegression, SVM, MLP, AdaBoost, XGBoost

In [ ]:
def run_loso_evaluation(features: Dict):
    """Run full LOSO evaluation for all classifiers."""
    classifiers = get_classifiers(SEED)
    param_grids = get_param_grids()
    subjects = sorted(features.keys())
    all_results = {name: [] for name in classifiers}

    for test_sid in tqdm(subjects, desc="LOSO folds"):
        X_train, y_train, X_test, y_test = prepare_fold(features, test_sid)
        has_both = len(np.unique(y_test)) == 2
        n_fog = int(np.sum(y_test == 1))

        log.info("Fold S%02d: test=%d (%d FoG), train=%d, both_classes=%s",
                 test_sid, len(y_test), n_fog, len(y_train), has_both)

        if len(y_test) == 0:
            continue

        X_train_p, X_test_p, sel_cols, pipes = preprocess_features(X_train, X_test, y_train, k=K_FEATURES)

        def _train_clf(clf_name):
            clf = get_classifiers(SEED)[clf_name]
            grid = param_grids[clf_name]
            m = train_and_evaluate_classifier(clf_name, clf, grid, X_train_p, y_train,
                                               X_test_p, y_test, seed=SEED,
                                               n_inner_cv=N_INNER_CV, n_search_iter=N_SEARCH_ITER)
            m["subject"] = test_sid
            m["has_both_classes"] = has_both
            return clf_name, m

        fold_results = Parallel(n_jobs=-1, verbose=0)(
            delayed(_train_clf)(name) for name in classifiers
        )

        for clf_name, m in fold_results:
            all_results[clf_name].append(m)

    return all_results

In [ ]:
# Step 3: LOSO evaluation
log.info("Running LOSO evaluation with %d classifiers...", len(get_classifiers(SEED)))
all_results = run_loso_evaluation(features)

## Classifier Comparison Results

In [ ]:
# Step 4: Print results
clf_rows = print_results_table(all_results)